# Data Science for Business - GAM on Ames Housing

*Session 6 · ISLP 7.5 Smoothing Splines · 7.7 GAMs*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
# Install packages that are not pre-installed on Colab
# !pip install "pygam>=0.10"

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import root_mean_squared_error

import statsmodels.formula.api as smf

from pygam import LinearGAM, s

In [ ]:
np.random.seed(42)
plt.style.use('fivethirtyeight')

## Problem description

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence. With 76 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this dataset challenges you to predict the final price of each home. More: <https://www.kaggle.com/c/house-prices-advanced-regression-techniques>

## Load data

Load data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_06/ameshousing.csv')

In [ ]:
data.head()

## Prepare data

Let us focus on the numerical variables only.

In [ ]:
data = data.select_dtypes(include='number')

In [ ]:
data.head()

Finally, we will split the data into predictors (*X*) and response (*y*) and into training (*X_train, y_train*) and test (*X_test, y_test*) sets.

In [ ]:
X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.head()

## Fit a linear regression as the baseline

To assess the performance of the GAM, we first fit a simple baseline model: a linear regression with two predictors, *YearBuilt* and *LotArea*.

In [ ]:
mod_01 = smf.ols(formula='SalePrice ~ YearBuilt + LotArea', data=pd.concat([X_train, y_train], axis=1))
mod_01 = mod_01.fit()
print(mod_01.summary2())

We evaluate all models in this notebook by their RMSE on the training and on the test set.

In [ ]:
y_pred_train = mod_01.predict(X_train)
y_pred_test = mod_01.predict(X_test)
print('Train RMSE:', root_mean_squared_error(y_train, y_pred_train))
print('Test RMSE: ', root_mean_squared_error(y_test, y_pred_test))

## Fit a GAM

A generalized additive model (GAM, ISLP 7.7) keeps the additive form $y = \beta_0 + f_1(x_1) + f_2(x_2) + \varepsilon$ but lets each $f_j$ be a smooth non-linear function.

Here, both $f_j$ are smoothing splines (ISLP 7.5), specified with `s()`. Each term has a roughness penalty λ (`lam`): a small λ gives a wiggly fit (risk of overfitting), a large λ pushes the function towards a straight line (ISLP 7.5.1). Strictly speaking, pygam approximates a smoothing spline: instead of a knot at every observation, it uses a penalised basis of 20 cubic spline functions (`n_splines=20`, `spline_order=3` by default).

pygam is not very user friendly: the terms refer to features by their column *index*, not their name. We therefore look up the index of each column by its name.

In [ ]:
X_train.columns

To start, we fit a GAM with almost no penalty (λ = 0.001).

In [ ]:
gam_wiggly = LinearGAM(s(X_train.columns.get_loc('LotArea')) + s(X_train.columns.get_loc('YearBuilt')), lam=0.001)
gam_wiggly.fit(X_train, y_train)

In [ ]:
y_pred_train = gam_wiggly.predict(X_train)
y_pred_test = gam_wiggly.predict(X_test)
print('Train RMSE:', root_mean_squared_error(y_train, y_pred_train))
print('Test RMSE: ', root_mean_squared_error(y_test, y_pred_test))

The training RMSE is clearly lower than that of the linear baseline, but its test RMSE is no better than the baseline (even slightly *higher*): the nearly unpenalized GAM overfits the training data.

Let's try to interpret the patterns the model has learned. For non-linear models, we typically do that visually. Below, we create partial dependence plots (incl. 95% confidence intervals) for all terms of the model (cf. ISLP Figs. 7.11/7.12). Each plot shows one feature's contribution to the predicted price, holding the other feature fixed. The dashed lines are the 95% bands; they are wide where data are sparse. Note how wiggly the curves are.

In [ ]:
for i, term in enumerate(gam_wiggly.terms):
    if term.isintercept:
        continue

    x_grid = gam_wiggly.generate_X_grid(term=i)
    pdep, confi = gam_wiggly.partial_dependence(term=i, X=x_grid, width=0.95)

    feature_name = X_train.columns[term.feature]

    plt.figure()
    plt.plot(x_grid[:, term.feature], pdep)
    plt.plot(x_grid[:, term.feature], confi, c='r', ls='--')
    plt.xlabel(feature_name)
    plt.ylabel('Partial effect on SalePrice')
    plt.title(f'{feature_name} (lambda = {term.lam[0]:.3g})')
    plt.show()

## Tune the smoothness penalty

λ controls the bias-variance trade-off (ISLP 7.5): a small λ gives a flexible fit with low bias but high variance (overfitting), a large λ gives a smooth, almost linear fit with low variance but higher bias. We choose λ by 5-fold cross-validation **on the training data** (ISLP 5.1.3), so the test set stays untouched.

We try 8 values between $10^{-3}$ and $10^{4}$, evenly spaced on a log scale, and use the same λ for both terms.

In [ ]:
lams = np.logspace(-3, 4, 8)
lams

Define the folds: the training data are shuffled and split into 5 folds.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

For every λ, the loop fits the GAM five times, each time on four folds, and computes the RMSE on the held-out fold. We store the mean CV RMSE per λ in `results`.

In [ ]:
results = []

for lam in lams:
    fold_rmse = []
    for train_idx, val_idx in kf.split(X_train):
        gam_cv = LinearGAM(s(X_train.columns.get_loc('LotArea')) + s(X_train.columns.get_loc('YearBuilt')), lam=lam)
        gam_cv.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
        y_pred_val = gam_cv.predict(X_train.iloc[val_idx])
        fold_rmse.append(root_mean_squared_error(y_train.iloc[val_idx], y_pred_val))

    results.append({'lam': lam, 'cv_rmse': np.mean(fold_rmse)})

results_df = pd.DataFrame(results)
results_df

Plot the CV RMSE against λ (log scale).

In [ ]:
plt.plot(results_df['lam'], results_df['cv_rmse'], marker='o')
plt.xscale('log')
plt.xlabel('lambda (log scale)')
plt.ylabel('CV RMSE')
plt.show()

The curve is U-shaped: with a very small λ the GAM overfits (high variance), with a very large λ the curves become too smooth (high bias). We pick the λ with the lowest CV RMSE.

In [ ]:
best_lam = results_df.loc[results_df['cv_rmse'].idxmin(), 'lam']
best_lam

Refit the GAM with the best λ on all training data and evaluate it once on the test set.

In [ ]:
gam_tuned = LinearGAM(s(X_train.columns.get_loc('LotArea')) + s(X_train.columns.get_loc('YearBuilt')), lam=best_lam)
gam_tuned.fit(X_train, y_train)

In [ ]:
y_pred_train = gam_tuned.predict(X_train)
y_pred_test = gam_tuned.predict(X_test)
print('Train RMSE:', root_mean_squared_error(y_train, y_pred_train))
print('Test RMSE: ', root_mean_squared_error(y_test, y_pred_test))

The tuned GAM has a slightly higher training RMSE than the wiggly GAM, but a clearly lower test RMSE, which is also lower than that of the linear baseline: allowing (smooth) non-linear effects pays off here.

Finally, let's visualize the effect of the tuning on the learned relationships. The curves are much smoother now.

In [ ]:
for i, term in enumerate(gam_tuned.terms):
    if term.isintercept:
        continue

    x_grid = gam_tuned.generate_X_grid(term=i)
    pdep, confi = gam_tuned.partial_dependence(term=i, X=x_grid, width=0.95)

    feature_name = X_train.columns[term.feature]

    plt.figure()
    plt.plot(x_grid[:, term.feature], pdep)
    plt.plot(x_grid[:, term.feature], confi, c='r', ls='--')
    plt.xlabel(feature_name)
    plt.ylabel('Partial effect on SalePrice')
    plt.title(f'{feature_name} (lambda = {term.lam[0]:.3g})')
    plt.show()

## Your turn!

**Your turn!**

1. Choose two other predictors, fit a `LinearGAM` with the tuned λ (`best_lam`), plot the partial dependences and compare the test RMSE with the models above. Which relationship is clearly non-linear?
2. The LotArea curve is sensitive to a few huge lots. Try to make it more robust, e.g., by fixing a larger λ for this term only (`s(..., lam=1000)`) or by using `np.log(LotArea)` as predictor. Does the test RMSE improve?

In [ ]:
# YOUR CODE HERE